# RQ7 Final Recommendation

**Research question:** Which model provides the best balance between predictive performance, interpretability, robustness, computational cost, and deployment suitability?

**Kaggle dataset source:** https://www.kaggle.com/datasets/shree0910/ai-and-data-science-job-market-dataset-20202026

**Dataset directory used in this notebook:** `/kaggle/input/datasets/shree0910/ai-and-data-science-job-market-dataset-20202026`

Outputs are saved under `/kaggle/working/ai_job_market_outputs/`.

In [ ]:

import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

RANDOM_STATE = 42

KAGGLE_DATASET_URL = "https://www.kaggle.com/datasets/shree0910/ai-and-data-science-job-market-dataset-20202026"
DATASET_DIR = "/kaggle/input/datasets/shree0910/ai-and-data-science-job-market-dataset-20202026"

OUTPUT_DIR = "/kaggle/working/ai_job_market_outputs"
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")

os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

print("Kaggle dataset source:", KAGGLE_DATASET_URL)
print("Dataset directory:", DATASET_DIR)

def find_dataset_file(dataset_dir=DATASET_DIR):
    supported_files = []
    for ext in ["*.csv", "*.xlsx", "*.xls"]:
        supported_files.extend(glob.glob(os.path.join(dataset_dir, "**", ext), recursive=True))

    if not supported_files:
        raise FileNotFoundError(
            f"No CSV or Excel file found under {dataset_dir}. Please attach the Kaggle dataset to this notebook."
        )

    supported_files = sorted(supported_files)
    print("Dataset file found:", supported_files[0])
    return supported_files[0]

def load_dataset():
    data_path = find_dataset_file()

    if data_path.lower().endswith(".csv"):
        df = pd.read_csv(data_path)
    else:
        df = pd.read_excel(data_path)

    df.columns = [c.strip() for c in df.columns]
    print("Dataset shape:", df.shape)
    display(df.head())
    return df

def get_feature_target(df):
    target_col = None

    for col in ["salary", "Salary", "SALARY"]:
        if col in df.columns:
            target_col = col
            break

    if target_col is None:
        possible = [c for c in df.columns if "salary" in c.lower()]
        if possible:
            target_col = possible[0]
        else:
            raise ValueError("No salary target column found. Please check the dataset columns.")

    drop_cols = [target_col]

    for id_col in ["job_id", "Job ID", "id", "ID"]:
        if id_col in df.columns:
            drop_cols.append(id_col)

    X = df.drop(columns=drop_cols)
    y = df[target_col]

    numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
    categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    return X, y, numeric_features, categorical_features, target_col

def make_preprocessor(numeric_features, categorical_features, scale_numeric=True):
    if scale_numeric:
        numeric_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
    else:
        numeric_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features)
        ],
        remainder="drop"
    )

    return preprocessor

def evaluate_regression_model(model, X_train, X_test, y_train, y_test, numeric_features, categorical_features, scale_numeric=True):
    pipe = Pipeline(steps=[
        ("preprocessor", make_preprocessor(
            numeric_features,
            categorical_features,
            scale_numeric=scale_numeric
        )),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    return pipe, mae, rmse, r2

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.titlesize": 13,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:

df = load_dataset()
X, y, numeric_features, categorical_features, target_col = get_feature_target(df)

print("Target column:", target_col)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)


In [ ]:

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=120, random_state=RANDOM_STATE, n_jobs=-1, max_depth=14),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "Extra Trees": ExtraTreesRegressor(n_estimators=120, random_state=RANDOM_STATE, n_jobs=-1, max_depth=14)
}

performance = []

for name, model in models.items():
    scale_numeric = name == "Linear Regression"

    pipe, mae, rmse, r2 = evaluate_regression_model(
        model,
        X_train,
        X_test,
        y_train,
        y_test,
        numeric_features,
        categorical_features,
        scale_numeric=scale_numeric
    )

    performance.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

perf_table = pd.DataFrame(performance)

def normalize_higher_better(series):
    if series.max() == series.min():
        return pd.Series([3] * len(series), index=series.index)
    return 1 + 4 * (series - series.min()) / (series.max() - series.min())

def normalize_lower_better(series):
    if series.max() == series.min():
        return pd.Series([3] * len(series), index=series.index)
    return 1 + 4 * (series.max() - series) / (series.max() - series.min())

decision = pd.DataFrame()
decision["Model"] = perf_table["Model"]
decision["Predictive_Performance"] = normalize_higher_better(perf_table["R2"])
decision["Cost_Efficiency"] = normalize_lower_better(perf_table["RMSE"])

interpretability_map = {
    "Linear Regression": 5.0,
    "Random Forest": 3.5,
    "Gradient Boosting": 3.0,
    "Extra Trees": 3.2
}

deployment_map = {
    "Linear Regression": 5.0,
    "Random Forest": 4.0,
    "Gradient Boosting": 3.7,
    "Extra Trees": 3.8
}

decision["Interpretability"] = decision["Model"].map(interpretability_map)
decision["Deployment_Suitability"] = decision["Model"].map(deployment_map)
decision["Robustness"] = (
    decision["Predictive_Performance"] + decision["Deployment_Suitability"]
) / 2

criteria_cols = [
    "Predictive_Performance",
    "Interpretability",
    "Robustness",
    "Cost_Efficiency",
    "Deployment_Suitability"
]

decision["Average_Score"] = decision[criteria_cols].mean(axis=1)
decision = decision.sort_values("Average_Score", ascending=False)

display(decision)

table_path = os.path.join(TABLE_DIR, "RQ7_Final_Recommendation.csv")
decision.to_csv(table_path, index=False)
print("Saved table:", table_path)

top_models = decision.head(3).copy()
labels = [c.replace("_", " ") for c in criteria_cols]

angles = np.linspace(0, 2 * np.pi, len(criteria_cols), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for _, row in top_models.iterrows():
    values = row[criteria_cols].tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=row["Model"])
    ax.fill(angles, values, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(["1", "2", "3", "4", "5"])
ax.set_ylim(0, 5)
ax.set_title("Figure 7. Final Model Trade-off Analysis", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.10), frameon=False)

fig.tight_layout()

figure_path = os.path.join(FIGURE_DIR, "RQ7_Final_Recommendation.pdf")
fig.savefig(figure_path, bbox_inches="tight")
plt.show()

print("Saved figure:", figure_path)
print("Recommended model:", decision.iloc[0]["Model"])
